# Auditoria de codigo_cvm por Ticker (2010–2025)

Este notebook audita, ticker a ticker, se o `codigo_cvm` atribuído está correto — e se há demonstrativos financeiros suficientes para usá-lo. A auditoria funciona como um funil: a cada camada, os tickers que já têm evidência suficiente são **descartados** (dados como resolvidos) e só os que sobram seguem para a camada seguinte. A cada etapa o notebook imprime:

- os tickers descartados nesta camada, com a evidência que sustentou a decisão;
- a tabela dos tickers que ainda restam a conferir.

Nenhum arquivo é gerado até a consolidação final.

**Camadas:**
1. **FCA** (Formulário Cadastral) — fonte primária: CNPJ ↔ ticker ↔ `codigo_cvm`, ano a ano.
2. **Cadastro geral de companhias abertas** (`cad_cia_aberta.csv`) — fonte de apoio para quem sobrou da Camada 1.
3. **Cobertura de demonstrativos (DFP/ITR)** — mesmo com `codigo_cvm` correto, verifica se existe demonstrativo para montar os indicadores no período.

**Sobre a Camada 3:** a exploração de formato, colunas e inconsistências dos arquivos DFP/ITR baixados **não é feita aqui**. Isso já foi tratado em `02_data_understanding.ipynb`. Este notebook apenas consome o resultado dessa exploração (um inventário de pares `codigo_cvm` x ano já extraído e validado) e valida que o contrato de entrada esperado (colunas, tipos) está sendo cumprido antes de seguir.

## Setup

Funções utilitárias usadas em todas as camadas: normalização de `codigo_cvm`, normalização de nome para comparação fuzzy, e um loader genérico para os CSVs anuais da CVM (todos seguem o mesmo layout: `;`, latin1, ano no nome do arquivo).

In [59]:
try:
    from rapidfuzz import fuzz
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "rapidfuzz", "--break-system-packages"])
    from rapidfuzz import fuzz

import pandas as pd
import glob
import os
import re
from pathlib import Path
import unicodedata

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)

caminho_atual = Path.cwd()
pasta_anterior = caminho_atual.parent


def normalizar_cvm(valor):
    """Remove zeros à esquerda e espaços, tratando codigo_cvm como número quando possível."""
    if valor is None:
        return None
    valor = str(valor).strip()
    if valor == "" or valor.upper() == "NAN":
        return None
    try:
        return str(int(valor))
    except ValueError:
        return valor


def normalizar_nome(nome):
    if not isinstance(nome, str):
        return ""
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("utf-8")
    nome = re.sub(r"[^A-Za-z0-9 ]", " ", nome)
    nome = re.sub(r"\s+", " ", nome).strip().upper()
    return nome


def melhor_score_nome(nome_base, nome_alvo):
    """Combina 3 métricas de similaridade (rapidfuzz) para lidar bem com nome
    curto/coloquial (ex: NOMRES do COTAHIST) vs razão social oficial e completa."""
    if not isinstance(nome_alvo, str) or nome_alvo == "" or not isinstance(nome_base, str):
        return 0
    a, b = normalizar_nome(nome_base), normalizar_nome(nome_alvo)
    return max(
        fuzz.token_set_ratio(a, b),
        fuzz.partial_ratio(a, b),
        fuzz.token_sort_ratio(a, b),
    )


def carregar_arquivos_cvm(padrao_glob):
    """Lê e concatena um conjunto de CSVs anuais da CVM (mesmo layout)."""
    arquivos = sorted(glob.glob(padrao_glob))
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para: {padrao_glob}")
    frames = []
    for arq in arquivos:
        df = pd.read_csv(arq, sep=";", encoding="latin1", dtype=str)
        df["arquivo_ano"] = os.path.basename(arq).split("_")[-1].replace(".csv", "")
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

## Passo 0 — Ponto de partida: a tabela completa de tickers

Carrega `ticker_and_code_cvm_mapping.csv` sem cruzar com nada ainda. Este é o "estado zero" do funil: todos os tickers começam como pendentes de verificação. A tabela completa fica visível aqui para referência das próximas etapas.

In [60]:
caminho_mapping = pasta_anterior / "data/interim/05_ticker_and_code_cvm_mapping.csv"

seu_df = pd.read_csv(caminho_mapping, sep=";", dtype=str)
seu_df["TICKER"] = seu_df["TICKER"].str.strip().str.upper()
seu_df["codigo_cvm"] = seu_df["codigo_cvm"].astype(str).str.strip()
seu_df["primeiro_ano_observado"] = seu_df["primeiro_ano_observado"].astype(int)
seu_df["ultimo_ano_observado"] = seu_df["ultimo_ano_observado"].astype(int)
seu_df["codigo_cvm_norm"] = seu_df["codigo_cvm"].apply(normalizar_cvm)

print(f"Total de tickers a auditar: {len(seu_df)}")
seu_df

Total de tickers a auditar: 769


,TICKER,codigo_cvm,codigo_isin,NOMRES,primeiro_ano_observado,ultimo_ano_observado,codigo_cvm_norm
0,AALR3,24058,BRAALRACNOR6,ALLIAR,2016,2025,24058
1,ABCB4,20958,BRABCBACNPR4,ABC BRASIL,2010,2025,20958
2,ABEV3,23264,BRABEVACNOR1,AMBEV S/A,2013,2025,23264
3,ABNB3,20028,BRABNBACNOR4,ABNOTE,2010,2010,20028
4,ABRE11,22551,BRABRECDAM15,ABRIL EDUCA,2011,2014,22551
...,...,...,...,...,...,...,...
764,WIZS3,23590,BRWIZSACNOR1,WIZ S.A.,2017,2023,23590
765,WLMM3,11070,BRWLMMACNOR6,WLM IND COM,2018,2025,11070
766,WLMM4,11070,BRWLMMACNPR3,WLM IND COM,2017,2025,11070
767,YDUQ3,21016,BRYDUQACNOR3,YDUQS PART,2019,2025,21016


## Passo 1 — Carregar referência FCA

Une `fca_cia_aberta_valor_mobiliario_AAAA.csv` (ticker ↔ CNPJ ↔ período de negociação) com `fca_cia_aberta_geral_AAAA.csv` (CNPJ ↔ `codigo_cvm` ↔ nome, ano a ano) para montar, por ticker, o conjunto de `codigo_cvm` que já esteve associado a ele — junto com o ano de cada associação (`candidatos_por_ano`). Isso permite checar depois se o código estava vigente **no período em que o ticker foi de fato negociado**, e não em qualquer ano da história do FCA.

In [61]:
diretorio_fca = pasta_anterior / "data/raw/06_fca_extracted"

fca_vm = carregar_arquivos_cvm(os.path.join(diretorio_fca, "fca_cia_aberta_valor_mobiliario_*.csv"))
fca_vm["CNPJ_Companhia"] = fca_vm["CNPJ_Companhia"].str.replace(r"\D", "", regex=True)
fca_vm["Codigo_Negociacao"] = fca_vm["Codigo_Negociacao"].str.strip().str.upper()
fca_vm["Data_Inicio_Negociacao"] = pd.to_datetime(fca_vm["Data_Inicio_Negociacao"], errors="coerce")
fca_vm["Data_Fim_Negociacao"] = pd.to_datetime(fca_vm["Data_Fim_Negociacao"], errors="coerce")
fca_vm = fca_vm[fca_vm["Codigo_Negociacao"].notna() & (fca_vm["Codigo_Negociacao"] != "")].copy()

fca_geral = carregar_arquivos_cvm(os.path.join(diretorio_fca, "fca_cia_aberta_geral_*.csv"))
fca_geral["CNPJ_Companhia"] = fca_geral["CNPJ_Companhia"].str.replace(r"\D", "", regex=True)
fca_geral["Codigo_CVM"] = fca_geral["Codigo_CVM"].astype(str).str.strip()
fca_geral["Data_Referencia"] = pd.to_datetime(fca_geral["Data_Referencia"], errors="coerce")
fca_geral["Nome_Empresarial"] = fca_geral["Nome_Empresarial"].str.strip()
fca_geral["Nome_Empresarial_Anterior"] = fca_geral["Nome_Empresarial_Anterior"].str.strip()

print("FCA valor mobiliário — linhas ticker x CNPJ x período:", len(fca_vm))
print("FCA geral — linhas CNPJ x codigo_cvm x ano:", len(fca_geral))

ref_cnpj_cvm = (
    fca_geral[["CNPJ_Companhia", "Codigo_CVM", "Nome_Empresarial", "Nome_Empresarial_Anterior", "Data_Referencia"]]
    .dropna(subset=["CNPJ_Companhia", "Codigo_CVM"])
    .drop_duplicates()
    .sort_values(["CNPJ_Companhia", "Data_Referencia"])
)
ref_cnpj_cvm["ano_referencia"] = ref_cnpj_cvm["Data_Referencia"].dt.year

n_cvm_por_cnpj = ref_cnpj_cvm.groupby("CNPJ_Companhia")["Codigo_CVM"].nunique()
print("CNPJs com mais de um codigo_cvm distinto ao longo do tempo:", (n_cvm_por_cnpj > 1).sum(), "de", len(n_cvm_por_cnpj))


def data_compativel(row):
    if pd.isna(row["Data_Referencia"]):
        return False
    inicio = row["Data_Inicio_Negociacao"]
    fim = row["Data_Fim_Negociacao"] if pd.notna(row["Data_Fim_Negociacao"]) else pd.Timestamp("2100-01-01")
    if pd.isna(inicio):
        return True
    return inicio <= row["Data_Referencia"] <= fim


fca_vm_reduzido = fca_vm[["CNPJ_Companhia", "Codigo_Negociacao", "Data_Inicio_Negociacao", "Data_Fim_Negociacao"]].copy()
ref_ticker_cvm = fca_vm_reduzido.merge(ref_cnpj_cvm, on="CNPJ_Companhia", how="left")
ref_ticker_cvm["compativel"] = ref_ticker_cvm.apply(data_compativel, axis=1)
ref_ticker_cvm = ref_ticker_cvm[ref_ticker_cvm["compativel"]].copy()


def nomes_unicos(serie_atual, serie_anterior):
    nomes = set(serie_atual.dropna()) | set(serie_anterior.dropna())
    return sorted(n for n in nomes if n)


ref_ticker_cvm_final = (
    ref_ticker_cvm
    .groupby("Codigo_Negociacao")
    .apply(lambda g: pd.Series({
        "codigo_cvm_fca": sorted(set(g["Codigo_CVM"])),
        "nome_empresarial_fca": nomes_unicos(g["Nome_Empresarial"], g["Nome_Empresarial_Anterior"]),
        "candidatos_por_ano": sorted(set(zip(g["ano_referencia"].dropna().astype(int), g["Codigo_CVM"]))),
    }))
    .reset_index()
    .rename(columns={"Codigo_Negociacao": "TICKER"})
)

print("\nTotal de tickers com alguma referência no FCA:", len(ref_ticker_cvm_final))

FCA valor mobiliário — linhas ticker x CNPJ x período: 4427
FCA geral — linhas CNPJ x codigo_cvm x ano: 10955
CNPJs com mais de um codigo_cvm distinto ao longo do tempo: 6 de 1260

Total de tickers com alguma referência no FCA: 752


Alguns tickers do FCA são preenchidos com valores residuais que não correspondem a tickers reais da B3 (ex.: `-`, `0`, `00000`, `999999`) — resíduos de cadastro que não afetam a auditoria, pois só entram no cruzamento se existir um `TICKER` correspondente na sua tabela.

## Passo 2 — Camada 1: cruzar com o FCA e separar o funil

Classifica cada ticker em:

- **ALTA** (descartado, resolvido): o `codigo_cvm` informado bate com um candidato do FCA para aquele ticker, **e** o ano desse candidato está dentro do intervalo `primeiro_ano_observado`–`ultimo_ano_observado` medido no COTAHIST. Essa exigência de ano evita validar um código que só passou a valer depois (ou parou de valer antes) do período em que o ticker foi observado.
- **PENDENTE** (segue no funil): sem referência FCA, ou nenhum candidato compatível no período bate com o `codigo_cvm` informado.

Abaixo: os descartados com a evidência que embasou a decisão, e a tabela dos que restam para a Camada 2.

In [62]:
auditoria = seu_df.merge(ref_ticker_cvm_final, on="TICKER", how="left")


def status_layer1(row):
    candidatos = row["candidatos_por_ano"]
    if not isinstance(candidatos, list) or len(candidatos) == 0:
        return "PENDENTE", None
    alvo = row["codigo_cvm_norm"]
    for ano, cvm in candidatos:
        dentro_do_periodo = row["primeiro_ano_observado"] <= ano <= row["ultimo_ano_observado"]
        if dentro_do_periodo and normalizar_cvm(cvm) == alvo:
            return "ALTA", ano
    return "PENDENTE", None


resultado_layer1 = auditoria.apply(status_layer1, axis=1, result_type="expand")
auditoria["status_layer1"] = resultado_layer1[0]
auditoria["ano_confirmacao_fca"] = resultado_layer1[1]

print(auditoria["status_layer1"].value_counts())

descartados_fca = auditoria[auditoria["status_layer1"] == "ALTA"]
pendentes_pos_fca = auditoria[auditoria["status_layer1"] == "PENDENTE"].copy()

print(f"\n--- Descartados via FCA ({len(descartados_fca)}) ---")
display(descartados_fca[["TICKER", "codigo_cvm", "ano_confirmacao_fca", "nome_empresarial_fca",
                          "primeiro_ano_observado", "ultimo_ano_observado"]])

print(f"\n--- Restam para a Camada 2 ({len(pendentes_pos_fca)}) ---")
display(pendentes_pos_fca[["TICKER", "codigo_cvm", "codigo_isin", "primeiro_ano_observado",
                            "ultimo_ano_observado", "codigo_cvm_fca", "nome_empresarial_fca"]])

status_layer1
ALTA        544
PENDENTE    225
Name: count, dtype: int64

--- Descartados via FCA (544) ---


,TICKER,codigo_cvm,ano_confirmacao_fca,nome_empresarial_fca,primeiro_ano_observado,ultimo_ano_observado
0,AALR3,24058,2017.0,"[ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A., CENTRO DE IMAGEM D...",2016,2025
1,ABCB4,20958,2010.0,"[BANCO ABC BRASIL S/A, BCO ABC BRASIL S.A., Banco ABC Ro...",2010,2025
2,ABEV3,23264,2014.0,"[AMBEV S.A., Ambev S.A., InBev Participações Societárias...",2013,2025
12,AELP3,19313,2010.0,"[AES ELPA S.A., AES ELPA SA, AES Elpa Ltda]",2010,2018
13,AERI3,25283,2021.0,[AERIS IND. E COM. DE EQUIP. PARA GER. DE ENG. S.A.],2020,2025
...,...,...,...,...,...,...
764,WIZS3,23590,2017.0,"[FPC PAR CORRETORA DE SEGUROS S.A., FPC Par Corretora de...",2017,2023
765,WLMM3,11070,2018.0,"[Supergasbras Indústria e Comércio S.A., WLM INDÚSTRIA E...",2018,2025
766,WLMM4,11070,2017.0,"[Supergasbras Indústria e Comércio S.A., WLM INDÚSTRIA E...",2017,2025
767,YDUQ3,21016,2019.0,"[ESTACIO PARTICIPACOES S.A., ESTACIO PARTICIPAÇÕES SA, Y...",2019,2025



--- Restam para a Camada 2 (225) ---


,TICKER,codigo_cvm,codigo_isin,primeiro_ano_observado,ultimo_ano_observado,codigo_cvm_fca,nome_empresarial_fca
3,ABNB3,20028,BRABNBACNOR4,2010,2010,NaN,NaN
4,ABRE11,22551,BRABRECDAM15,2011,2014,NaN,NaN
5,ABRE3,22551,BRABREACNOR9,2014,2015,NaN,NaN
6,ABYA3,20206,BRABYAACNOR3,2010,2010,NaN,NaN
7,ACGU3,20940,BRACGUACNOR6,2010,2010,NaN,NaN
...,...,...,...,...,...,...,...
743,VIVO4,17671,BRVIVOACNPR8,2010,2011,NaN,NaN
749,VTLM3,21725,BRVTLMACNOR3,2016,2016,NaN,NaN
756,WDCN3,25895,BRWDCNACNOR2,2025,2025,NaN,NaN
761,WISA3,14397,BRWISAACNOR4,2011,2012,NaN,NaN


## Passo 3 — Camada 2: cruzar os pendentes com `cad_cia_aberta`

Para quem sobrou da Camada 1, verifica se o `codigo_cvm` informado existe no cadastro geral de companhias abertas da CVM e, se existir, compara `NOMRES` (nome curto do COTAHIST) com `DENOM_SOCIAL` (razão social oficial) usando o melhor entre 3 métricas de similaridade (`token_set_ratio`, `partial_ratio`, `token_sort_ratio`). Isso é necessário porque nomes curtos/coloquiais como "AMBEV" batem mal com razões sociais completas como "COMPANHIA DE BEBIDAS DAS AMÉRICAS-AMBEV" em métricas simples de distância de string.

Três desfechos:
- **ALTA_VIA_CADASTRO** (descartado): código existe e o nome bate (score ≥ `LIMIAR_NOME`).
- **REVISAR_NOME_DIVERGENTE** (segue no funil): código existe, mas o nome não bate.
- **REVISAR_CODIGO_INEXISTENTE** (segue no funil): o `codigo_cvm` nem existe no cadastro — sinal forte de erro de digitação/transcrição, ou caso atípico fora do registro padrão (ex.: BDR/unit).

**Nota:** o cadastro é reduzido a um registro por `codigo_cvm` mantendo a razão social mais recente (`keep="last"`). Isso significa que tickers antigos, cujo nome no COTAHIST reflete uma razão social já trocada há anos, tendem a cair em `REVISAR_NOME_DIVERGENTE` mesmo com o `codigo_cvm` correto — vale ter isso em mente ao revisar a lista, e considerar comparar contra o histórico de nomes se o volume de divergências por esse motivo for grande.

In [63]:
caminho_cad = pasta_anterior / "data/raw/06_cad_cia_aberta.csv"

cad = pd.read_csv(caminho_cad, sep=";", encoding="latin1", dtype=str)
cad.columns = [c.strip() for c in cad.columns]
cad["CD_CVM_NORM"] = cad["CD_CVM"].astype(str).str.strip().apply(normalizar_cvm)
cad["DENOM_SOCIAL"] = cad["DENOM_SOCIAL"].str.strip()
cad_unico = cad.drop_duplicates(subset="CD_CVM_NORM", keep="last")
print("Companhias únicas no cadastro:", len(cad_unico))

LIMIAR_NOME = 80

pendentes_pos_fca = pendentes_pos_fca.merge(
    cad_unico[["CD_CVM_NORM", "DENOM_SOCIAL"]],
    left_on="codigo_cvm_norm", right_on="CD_CVM_NORM", how="left"
)
pendentes_pos_fca["existe_no_cadastro"] = pendentes_pos_fca["DENOM_SOCIAL"].notna()
pendentes_pos_fca["score_nome_cadastro"] = pendentes_pos_fca.apply(
    lambda r: melhor_score_nome(r.get("NOMRES"), r["DENOM_SOCIAL"]) if r["existe_no_cadastro"] else 0,
    axis=1,
)


def status_layer2(row):
    if not row["existe_no_cadastro"]:
        return "REVISAR_CODIGO_INEXISTENTE"
    if row["score_nome_cadastro"] >= LIMIAR_NOME:
        return "ALTA_VIA_CADASTRO"
    return "REVISAR_NOME_DIVERGENTE"


pendentes_pos_fca["status_layer2"] = pendentes_pos_fca.apply(status_layer2, axis=1)
print(pendentes_pos_fca["status_layer2"].value_counts())

descartados_cadastro = pendentes_pos_fca[pendentes_pos_fca["status_layer2"] == "ALTA_VIA_CADASTRO"]
pendentes_pos_cadastro = pendentes_pos_fca[pendentes_pos_fca["status_layer2"] != "ALTA_VIA_CADASTRO"].copy()

print(f"\n--- Descartados via cadastro ({len(descartados_cadastro)}) ---")
display(descartados_cadastro[["TICKER", "codigo_cvm", "DENOM_SOCIAL", "score_nome_cadastro"]]
        .sort_values("score_nome_cadastro"))

print(f"\n--- Restam para revisão manual / Camada 3 ({len(pendentes_pos_cadastro)}) ---")
display(pendentes_pos_cadastro[["TICKER", "codigo_cvm", "status_layer2", "DENOM_SOCIAL", "score_nome_cadastro"]]
        .sort_values("score_nome_cadastro"))

Companhias únicas no cadastro: 2566
status_layer2
ALTA_VIA_CADASTRO             140
REVISAR_NOME_DIVERGENTE        84
REVISAR_CODIGO_INEXISTENTE      1
Name: count, dtype: int64

--- Descartados via cadastro (140) ---


,TICKER,codigo_cvm,DENOM_SOCIAL,score_nome_cadastro
10,AGEI3,21911,AGRE EMPREENDIMENTOS IMOBILIÁRIOS SA,80.000000
51,CAFE4,2062,CAFE SOLUVEL BRASILIA SA,80.000000
195,TCSL4,17639,TIM PARTICIPAÇÕES SA,80.000000
114,IMCH3,22438,"INTERNATIONAL MEAL COMPANY HOLDINGS, SA",80.000000
202,TLPP3,17671,TELEFÔNICA BRASIL S.A.,80.000000
194,TCSL3,17639,TIM PARTICIPAÇÕES SA,80.000000
203,TLPP4,17671,TELEFÔNICA BRASIL S.A.,80.000000
145,NORD3,9083,NORDON INDUSTRIAS METALURGICAS S.A.,82.352941
197,TENE5,11215,TECBLU - TECELAGEM BLUMENAU S/A.,83.333333
198,TENE7,11215,TECBLU - TECELAGEM BLUMENAU S/A.,83.333333



--- Restam para revisão manual / Camada 3 (85) ---


,TICKER,codigo_cvm,status_layer2,DENOM_SOCIAL,score_nome_cadastro
156,PPLA11,80152,REVISAR_CODIGO_INEXISTENTE,NaN,0.000000
220,VIVO4,17671,REVISAR_NOME_DIVERGENTE,TELEFÔNICA BRASIL S.A.,25.000000
219,VIVO3,17671,REVISAR_NOME_DIVERGENTE,TELEFÔNICA BRASIL S.A.,25.000000
82,ECOD3,20354,REVISAR_NOME_DIVERGENTE,TERRA SANTA AGRO S.A.,28.571429
200,TIBR5,11398,REVISAR_NOME_DIVERGENTE,TRONOX PIGMENTOS DO BRASIL S.A.,30.000000
201,TIBR6,11398,REVISAR_NOME_DIVERGENTE,TRONOX PIGMENTOS DO BRASIL S.A.,30.000000
129,LVTC3,25895,REVISAR_NOME_DIVERGENTE,LIVETECH DA BAHIA INDÚSTRIA E COMÉRCIO S.A.,33.333333
0,ABNB3,20028,REVISAR_NOME_DIVERGENTE,VALID SOLUÇÕES S.A.,33.333333
148,OHLB3,19771,REVISAR_NOME_DIVERGENTE,ARTERIS S.A.,33.333333
222,WDCN3,25895,REVISAR_NOME_DIVERGENTE,LIVETECH DA BAHIA INDÚSTRIA E COMÉRCIO S.A.,33.333333


## Passo 4 — Consolidar `status_final`

Junta as duas camadas num único dataframe `auditoria_final`, com `status_final` para todos os tickers da tabela original. Ainda sem gerar arquivo — só o print do resumo geral e da lista consolidada de quem precisa de checagem manual.

In [64]:
mapa_layer2 = pendentes_pos_fca.set_index("TICKER")[["status_layer2", "DENOM_SOCIAL", "score_nome_cadastro"]]

auditoria_final = auditoria.merge(mapa_layer2, on="TICKER", how="left")


def status_consolidado(row):
    if row["status_layer1"] == "ALTA":
        return "ALTA_VIA_FCA"
    return row["status_layer2"]


auditoria_final["status_final"] = auditoria_final.apply(status_consolidado, axis=1)

print(auditoria_final["status_final"].value_counts())

n_revisar = auditoria_final["status_final"].isin(["REVISAR_NOME_DIVERGENTE", "REVISAR_CODIGO_INEXISTENTE"]).sum()
print(f"\nValidados automaticamente: {len(auditoria_final) - n_revisar} de {len(auditoria_final)} "
      f"({(len(auditoria_final) - n_revisar) / len(auditoria_final):.1%})")

colunas_master = ["TICKER", "codigo_cvm", "codigo_isin", "primeiro_ano_observado",
                   "ultimo_ano_observado", "status_final", "DENOM_SOCIAL", "score_nome_cadastro"]
colunas_master = [c for c in colunas_master if c in auditoria_final.columns]

revisar = auditoria_final[auditoria_final["status_final"].isin(
    ["REVISAR_NOME_DIVERGENTE", "REVISAR_CODIGO_INEXISTENTE"]
)].sort_values("score_nome_cadastro")

print(f"\n--- {len(revisar)} tickers para checagem manual pontual ---")
revisar[colunas_master]

status_final
ALTA_VIA_FCA                  544
ALTA_VIA_CADASTRO             140
REVISAR_NOME_DIVERGENTE        84
REVISAR_CODIGO_INEXISTENTE      1
Name: count, dtype: int64

Validados automaticamente: 684 de 769 (88.9%)

--- 85 tickers para checagem manual pontual ---


,TICKER,codigo_cvm,codigo_isin,primeiro_ano_observado,ultimo_ano_observado,status_final,DENOM_SOCIAL,score_nome_cadastro
553,PPLA11,80152,BRPPLAUNT007,2017,2022,REVISAR_CODIGO_INEXISTENTE,NaN,0.000000
743,VIVO4,17671,BRVIVOACNPR8,2010,2011,REVISAR_NOME_DIVERGENTE,TELEFÔNICA BRASIL S.A.,25.000000
742,VIVO3,17671,BRVIVOACNOR1,2010,2011,REVISAR_NOME_DIVERGENTE,TELEFÔNICA BRASIL S.A.,25.000000
272,ECOD3,20354,BRECODACNOR8,2010,2011,REVISAR_NOME_DIVERGENTE,TERRA SANTA AGRO S.A.,28.571429
690,TIBR5,11398,BRTIBRACNPA3,2010,2015,REVISAR_NOME_DIVERGENTE,TRONOX PIGMENTOS DO BRASIL S.A.,30.000000
691,TIBR6,11398,BRTIBRACNPB1,2010,2014,REVISAR_NOME_DIVERGENTE,TRONOX PIGMENTOS DO BRASIL S.A.,30.000000
447,LVTC3,25895,BRLVTCACNOR4,2021,2025,REVISAR_NOME_DIVERGENTE,LIVETECH DA BAHIA INDÚSTRIA E COMÉRCIO S.A.,33.333333
3,ABNB3,20028,BRABNBACNOR4,2010,2010,REVISAR_NOME_DIVERGENTE,VALID SOLUÇÕES S.A.,33.333333
513,OHLB3,19771,BROHLBACNOR6,2010,2012,REVISAR_NOME_DIVERGENTE,ARTERIS S.A.,33.333333
756,WDCN3,25895,BRWDCNACNOR2,2025,2025,REVISAR_NOME_DIVERGENTE,LIVETECH DA BAHIA INDÚSTRIA E COMÉRCIO S.A.,33.333333


## Passo 5 — Inventário de `codigo_cvm` presentes nos arquivos DFP/ITR

Antes de cruzar com os tickers já validados, construímos um inventário de
todos os pares `(codigo_cvm, ano)` que efetivamente existem nos arquivos
parquet baixados. Usamos os arquivos "principais" (`dfp_cia_aberta_AAAA-AAAA.parquet`
e `itr_cia_aberta_AAAA-AAAA.parquet`) — eles contêm `CD_CVM`, `DT_REFER` e
`DENOM_CIA` para toda companhia com demonstrativo naquele ano, sem precisar
ler os arquivos grandes de cada tipo de demonstrativo (BPA, BPP, DRE, etc.),
já que o conjunto de `codigo_cvm` é o mesmo entre eles.

In [65]:
import duckdb

pasta_interim = pasta_anterior / "data/interim"

arquivos_principais = {
    "dfp": sorted((pasta_interim / "02_dfp_concatenated").glob("dfp_cia_aberta_*.parquet"))[0],
    "itr": sorted((pasta_interim / "02_itr_concatenated").glob("itr_cia_aberta_*.parquet"))[0],
}

# Confere que peguei o arquivo "principal" certo (o mais leve, sem sufixo de demonstrativo)
for origem, arq in arquivos_principais.items():
    print(origem, "->", arq.name)

dfp -> dfp_cia_aberta_2010-2025.parquet
itr -> itr_cia_aberta_2011-2025.parquet


Extrai, de cada arquivo, os pares únicos `(codigo_cvm, ano, denom_cia)`.
`ano` é derivado de `DT_REFER` (data de referência do demonstrativo).

In [66]:
def extrair_codigos_cvm(arquivo_parquet: Path, origem: str) -> pd.DataFrame:
    df = duckdb.sql(f"""
        SELECT DISTINCT
            CD_CVM AS codigo_cvm,
            CAST(SUBSTR(DT_REFER, 1, 4) AS INTEGER) AS ano,
            DENOM_CIA AS denom_cia
        FROM '{arquivo_parquet}'
    """).df()
    df["codigo_cvm"] = df["codigo_cvm"].astype(str).apply(normalizar_cvm)
    df["origem"] = origem
    return df


codigos_cvm_parquet = pd.concat(
    [extrair_codigos_cvm(arq, origem) for origem, arq in arquivos_principais.items()],
    ignore_index=True,
)

print(f"Total de pares (codigo_cvm, ano, origem): {len(codigos_cvm_parquet)}")
codigos_cvm_parquet.head()

Total de pares (codigo_cvm, ano, origem): 20999


,codigo_cvm,ano,denom_cia,origem
0,14451,2010,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,dfp
1,14613,2010,NEUMARKT TRADE AND FINANCIAL CENTER S/A,dfp
2,15334,2010,BOMBRIL HOLDING SA,dfp
3,16195,2010,AMERICEL SA,dfp
4,16551,2010,WTC RIO EMPREEND. E PARTICIPAÇÕES S.A.,dfp


### Códigos CVM únicos

Lista consolidada de todo `codigo_cvm` que aparece em pelo menos um
demonstrativo (DFP ou ITR), independentemente do ano — é o universo total
de companhias com dados financeiros disponíveis nos parquet.

In [67]:
codigos_cvm_unicos = (
    codigos_cvm_parquet
    .groupby("codigo_cvm")
    .agg(
        primeiro_ano_dfp_itr=("ano", "min"),
        ultimo_ano_dfp_itr=("ano", "max"),
        anos_disponiveis=("ano", lambda s: sorted(set(s))),
        origens=("origem", lambda s: sorted(set(s))),
        denom_cia_mais_recente=("denom_cia", "last"),
    )
    .reset_index()
)

print(f"Total de codigo_cvm únicos nos arquivos DFP/ITR: {len(codigos_cvm_unicos)}")
codigos_cvm_unicos

Total de codigo_cvm únicos nos arquivos DFP/ITR: 1228


,codigo_cvm,primeiro_ano_dfp_itr,ultimo_ano_dfp_itr,anos_disponiveis,origens,denom_cia_mais_recente
0,1023,2010,2025,"[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2...","[dfp, itr]",BCO BRASIL S.A.
1,10456,2010,2025,"[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2...","[dfp, itr]",ALPARGATAS S.A.
2,10472,2010,2023,"[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2...","[dfp, itr]",SARAIVA LIVREIROS S.A. - EM RECUPERAÇÃO JUDICIAL
3,10561,2010,2025,"[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2...","[dfp, itr]",SEIVA S.A. - FLORESTAS E INDÚSTRIAS
4,10596,2010,2013,"[2010, 2011, 2012, 2013]","[dfp, itr]",SERGEN SERVS GERAIS DE ENG SA
...,...,...,...,...,...,...
1223,9717,2010,2025,"[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2...","[dfp, itr]",PORTUENSE FERRAGENS S/A
1224,9784,2010,2011,"[2010, 2011]","[dfp, itr]",PRONOR PETROQUIMICA SA
1225,9857,2010,2012,"[2010, 2011, 2012]","[dfp, itr]",QGN PARTICIPAÇÕES SA
1226,9954,2010,2025,"[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2...","[dfp, itr]",ALFA HOLDINGS S.A.


## Passo 6 — Cruzar tickers validados com a cobertura DFP/ITR

Para cada ticker com `status_final` já resolvido (`ALTA_VIA_FCA` ou
`ALTA_VIA_CADASTRO`), verificamos:

1. se o `codigo_cvm` existe em algum ano nos parquet (`tem_demonstrativo`);
2. se existe demonstrativo **dentro do período em que o ticker foi
   negociado** (`primeiro_ano_observado`–`ultimo_ano_observado`) — o que
   importa de fato para montar indicadores;
3. quantos anos de demonstrativo existem nesse período
   (`anos_cobertos_no_periodo`).

Isso separa dois problemas diferentes: `codigo_cvm` correto mas **sem**
demonstrativo no período (situação real de dados faltantes) versus
`codigo_cvm` correto e coberto (pronto para uso).

In [68]:
validados = auditoria_final[
    auditoria_final["status_final"].isin(["ALTA_VIA_FCA", "ALTA_VIA_CADASTRO"])
].copy()

mapa_anos_por_cvm = codigos_cvm_unicos.set_index("codigo_cvm")["anos_disponiveis"].to_dict()


def cobertura_dfp_itr(row):
    anos_disponiveis = mapa_anos_por_cvm.get(row["codigo_cvm_norm"])
    if anos_disponiveis is None:
        return pd.Series({
            "tem_demonstrativo": False,
            "anos_cobertos_no_periodo": [],
            "qtd_anos_cobertos_no_periodo": 0,
        })

    anos_no_periodo = [
        ano for ano in anos_disponiveis
        if row["primeiro_ano_observado"] <= ano <= row["ultimo_ano_observado"]
    ]

    return pd.Series({
        "tem_demonstrativo": True,
        "anos_cobertos_no_periodo": anos_no_periodo,
        "qtd_anos_cobertos_no_periodo": len(anos_no_periodo),
    })


validados = pd.concat([validados, validados.apply(cobertura_dfp_itr, axis=1)], axis=1)

print("Tem ao menos 1 ano de demonstrativo no período observado:",
      (validados["qtd_anos_cobertos_no_periodo"] > 0).sum(), "de", len(validados))

print("\nSem NENHUM demonstrativo (mesmo fora do período):",
      (~validados["tem_demonstrativo"]).sum())

print("\nTem demonstrativo, mas fora do período observado do ticker:",
      ((validados["tem_demonstrativo"]) & (validados["qtd_anos_cobertos_no_periodo"] == 0)).sum())

Tem ao menos 1 ano de demonstrativo no período observado: 673 de 684

Sem NENHUM demonstrativo (mesmo fora do período): 11

Tem demonstrativo, mas fora do período observado do ticker: 0


### Casos sem cobertura — merecem checagem

Tickers com `codigo_cvm` validado, mas sem demonstrativo disponível no
período em que foram negociados. Esses são os que, na prática, ficarão sem
indicador fundamentalista mesmo com o código correto.

In [69]:
sem_cobertura = validados[validados["qtd_anos_cobertos_no_periodo"] == 0][
    ["TICKER", "codigo_cvm", "status_final", "primeiro_ano_observado",
     "ultimo_ano_observado", "tem_demonstrativo"]
].sort_values("TICKER")

print(f"--- {len(sem_cobertura)} tickers sem cobertura DFP/ITR no período observado ---")
sem_cobertura

--- 11 tickers sem cobertura DFP/ITR no período observado ---


,TICKER,codigo_cvm,status_final,primeiro_ano_observado,ultimo_ano_observado,tem_demonstrativo
6,ABYA3,20206,ALTA_VIA_CADASTRO,2010,2010,False
7,ACGU3,20940,ALTA_VIA_CADASTRO,2010,2010,False
17,AGEI3,21911,ALTA_VIA_CADASTRO,2010,2010,False
55,AVIL3,108,ALTA_VIA_CADASTRO,2010,2011,False
359,GVTT3,20117,ALTA_VIA_CADASTRO,2010,2010,False
422,KSSA3,20249,ALTA_VIA_CADASTRO,2010,2010,False
452,MARI3,20761,ALTA_VIA_CADASTRO,2010,2010,False
459,MEDI3,20273,ALTA_VIA_CADASTRO,2010,2010,False
719,TVIT3,21768,ALTA_VIA_CADASTRO,2010,2010,False
761,WISA3,14397,ALTA_VIA_CADASTRO,2011,2012,False


## Passo 7 — Exportação final

Consolida tudo (status de auditoria + cobertura DFP/ITR) e exporta:

- `auditoria_codigo_cvm_completa.csv`: todos os 769 tickers, com status
  final e indicadores de cobertura.
- `auditoria_codigo_cvm_para_revisao.csv`: apenas os casos que restaram
  para checagem manual (nome divergente / código inexistente), do mais
  duvidoso ao menos duvidoso.
- `codigo_cvm_unicos_dfp_itr.csv`: lista de referência com todo `codigo_cvm`
  disponível nos parquet — útil depois para localizar rapidamente os dados
  de uma empresa sem precisar escanear os arquivos grandes de novo.

In [70]:
pasta_saida = pasta_anterior / "data/interim"

# 1. Junta cobertura DFP/ITR de volta no dataframe completo (quem não foi
#    validado nesta etapa fica com NaN nessas colunas — é esperado)
colunas_cobertura = ["TICKER", "tem_demonstrativo", "qtd_anos_cobertos_no_periodo"]

auditoria_export = auditoria_final.merge(
    validados[colunas_cobertura], on="TICKER", how="left"
)

# 2. Arquivo completo
colunas_finais = [
    "TICKER", "codigo_cvm", "codigo_isin", "primeiro_ano_observado",
    "ultimo_ano_observado", "status_final", "DENOM_SOCIAL",
    "score_nome_cadastro", "tem_demonstrativo", "qtd_anos_cobertos_no_periodo",
]
colunas_finais = [c for c in colunas_finais if c in auditoria_export.columns]

auditoria_export[colunas_finais].to_csv(
    pasta_saida / "07_audit/complete_code_cvm.csv", sep=";", index=False
)

# 3. Arquivo apenas com pendências de revisão manual
revisar_export = auditoria_export[
    auditoria_export["status_final"].isin(["REVISAR_NOME_DIVERGENTE", "REVISAR_CODIGO_INEXISTENTE"])
].sort_values("score_nome_cadastro")

revisar_export[colunas_finais].to_csv(
    pasta_saida / "07_audit/revision_code_cvm.csv", sep=";", index=False
)

# 4. Lista de referência de codigo_cvm disponíveis nos parquet
codigos_cvm_unicos.to_csv(
    pasta_saida / "07_audit/unique_code_cvm_for_dfp-itr.csv", sep=";", index=False
)

print("Exportado:")
print(" -", pasta_saida / "07_audit/complete_code_cvm.csv", f"({len(auditoria_export)} linhas)")
print(" -", pasta_saida / "07_audit/revision_code_cvm.csv", f"({len(revisar_export)} linhas)")
print(" -", pasta_saida / "07_audit/unique_code_cvm_for_dfp-itr.csv", f"({len(codigos_cvm_unicos)} linhas)")

Exportado:
 - c:\Users\paulo\Desktop\Python\brazilian_financial_database\data\interim\07_audit\complete_code_cvm.csv (769 linhas)
 - c:\Users\paulo\Desktop\Python\brazilian_financial_database\data\interim\07_audit\revision_code_cvm.csv (85 linhas)
 - c:\Users\paulo\Desktop\Python\brazilian_financial_database\data\interim\07_audit\unique_code_cvm_for_dfp-itr.csv (1228 linhas)
